# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [ ]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Tener que reentrenar o reconstruir el modelo en cada ejecución: el objeto ejecutable viaja tal cual y no depende del notebook que lo creó. |
| Versión del esquema | **manifest.json** (`schema_version`) | Que el cargador lea un manifiesto con un formato que no entiende (claves renombradas o nuevas) sin darse cuenta. |
| Orden y nombres de features | **manifest.json** (`feature_names`) y **código** (`FEATURE_NAMES` en `preprocess.py`) | Que el vector llegue con columnas cambiadas de sitio (p. ej. `density` y `alcohol`) y el modelo prediga en silencio con datos mal colocados. |
| Versión del preprocesado | **manifest.json** (`preprocessing_version`) y **código** (`PREPROCESSING_VERSION`) | Usar un modelo con un preprocesado distinto del que aplica el código actual. |
| Etiquetas de salida permitidas | **manifest.json** (`output_labels`) y **código** (`QualityBand` en el contrato de salida) | Que la interfaz reciba una categoría que no sabe tratar, como `unknown`. |
| Versión del modelo | **manifest.json** (`model_version`), que además se copia en cada predicción | No poder saber qué modelo generó una predicción (trazabilidad, comparación y vuelta atrás). |
| Tipo de estimador | **manifest.json** (`estimator_type`), comprobado en **código** contra `type(estimator).__name__` | Que `model.joblib` se sustituya por otro objeto (otro modelo o un archivo equivocado) sin que el manifiesto lo refleje. |

Regla general: el binario sólo contiene lo que se ejecuta; todo lo que sirve para **decidir si se puede ejecutar** va en el manifiesto, que es texto legible y se valida antes de abrir el binario; las comprobaciones viven en el código y en las pruebas.

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [ ]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "wine-quality-bundle-v1",
    "model_version": legacy_model_version,
    "preprocessing_version": "wine-red-features-v1",
    "feature_names": feature_names,
    # Las categorías que devuelve el clasificador de la semana 3 y que la
    # interfaz sabe mostrar; "low/medium/high" eran etiquetas provisionales.
    "output_labels": ["needs_review", "acceptable", "excellent"],
    "estimator_type": "RandomForestClassifier",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]
assert len(set(manifest_draft["feature_names"])) == 11  # sin nombres repetidos
assert "TODO" not in json.dumps(manifest_draft)
assert set(manifest_draft["output_labels"]) == {"needs_review", "acceptable", "excellent"}

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida y no quedan TODO ni etiquetas provisionales.")

### Decisiones e invariantes del manifiesto

**Valores elegidos**

- `schema_version = "wine-quality-bundle-v1"`: versión del *formato* del bundle, independiente de la del modelo. Si cambian las claves del manifiesto, sube.
- `preprocessing_version = "wine-red-features-v1"`: la misma constante que usa `preprocess.py` en la semana 3.
- `output_labels = ["needs_review", "acceptable", "excellent"]`: las clases del clasificador de la semana 3 y los valores de `QualityBand`.
- `estimator_type = "RandomForestClassifier"`: el tipo del estimador del payload legado.

**Qué debe pasar si…**

1. **Falta `manifest.json`:** el cargador se detiene con `FileNotFoundError` antes de abrir `model.joblib`; sin manifiesto no hay forma de saber si el binario es compatible. El CLI responde con código 2 y no escribe CSV.
2. **Hay features desconocidas o en otro orden:** el manifiesto no se acepta (`feature_names` debe ser exactamente `FEATURE_NAMES`, mismo orden) y no se llega a cargar el binario. Error con la lista recibida.
3. **El modelo devuelve una etiqueta fuera de `output_labels`:** `infer_wine_quality` lanza `ValueError` con la etiqueta recibida y las permitidas; el CLI aborta sin escribir predicciones parciales.
4. **Una fila del CSV no cumple el contrato:** falla `WineQualityRequest` al leer el CSV, antes incluso de cargar el bundle; el mensaje indica el número de fila y el campo.

**Tres invariantes que debe comprobar el cargador**

1. `manifest.json` existe, tiene exactamente las seis claves (sin extras) y `schema_version` es la que entiende el código. Se comprueba **antes** de deserializar `model.joblib`, porque deserializar un joblib/pickle puede ejecutar código.
2. `feature_names == FEATURE_NAMES` (11 nombres, mismo orden, sin `sample_id`) y `preprocessing_version == PREPROCESSING_VERSION`.
3. `model.joblib` contiene un estimador con `predict` y `predict_proba` cuyo tipo coincide con `estimator_type`; y toda etiqueta que devuelva pertenece a `output_labels`.

**Dos entradas inválidas y dónde fallan**

- Manifiesto con `feature_names` invertido → falla en la **carga del bundle**, al validar el manifiesto.
- Fila con `ph = 99` → falla en el **contrato de entrada**, al leer el CSV, antes de predecir.

## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | Carga del bundle: validación del manifiesto, antes de abrir `model.joblib` | `feature_names no coincide con FEATURE_NAMES: (...)` con el orden recibido | No |
| Cambia `preprocessing_version` | Carga del bundle: comparación del manifiesto con `PREPROCESSING_VERSION` del código | `preprocessing_version 'x' no coincide con el preprocesado del código ('wine-red-features-v1')` | No |
| El modelo devuelve `unknown` | Validación de la salida en `infer_wine_quality` (contra `output_labels`) | `El modelo devolvió 'unknown', que no está en (...)` | No: el CLI aborta con código 2 antes de escribir |
| La petición incluye `ph = 99` | Contrato de entrada (`WineQualityRequest`) al leer el CSV, antes de cargar el bundle | `fila 2: ... ph: Input should be less than or equal to 4.5` | No |

Si el intercambio de `density` y `alcohol` está en las **columnas del CSV** y no en el manifiesto, el CLI también lo rechaza porque exige la cabecera exacta `sample_id + FEATURE_NAMES`.

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
- ¿Aceptarás campos desconocidos en el manifiesto?
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?

**Decisiones:**

- **Obligatorios:** las seis claves del manifiesto, sin valores por defecto: un manifiesto incompleto es un error, no algo que se rellena en silencio. En el CSV, `sample_id` y las 11 features son obligatorias.
- **Campos desconocidos:** no se aceptan (`extra="forbid"`). Añadir un campo al manifiesto implica subir `schema_version`.
- **Orden de validación:** sí, primero el manifiesto y después el binario. Evita deserializar un artefacto incompatible y da un mensaje más claro.
- **Escritura del CSV:** sólo al final, después de validar todas las filas y obtener todas las predicciones. Se acumulan en memoria y se escribe una sola vez.
- **Mensaje:** `error: <qué ha fallado y dónde>` en stderr y código de salida 2, sin traceback: número de fila y campo para el CSV; ruta del bundle y clave o archivo para el artefacto.
- **Segunda fila inválida:** se aborta todo con código 2 y no se escribe ningún CSV, ni siquiera con la primera fila correcta. Una salida parcial podría tomarse por completa.

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.